# 01 — Setup & Verification

This notebook verifies that all components work:
1. MGSM dataset loads correctly for all 5 languages
2. Gemma 3 4B IT model loads and generates answers
3. Gemma Scope 2 SAEs load via SAELens
4. nnsight can extract activations
5. Baseline accuracy on a small subset

**Run this on Colab with A100 GPU.**

## 0. Install Dependencies

In [1]:
# Run this cell on Colab, then RESTART RUNTIME before continuing
# (Runtime → Restart session)
# Note: we do NOT install 'datasets' — MGSM is loaded directly from source to avoid dependency conflicts
!pip install -q transformers sae-lens nnsight python-dotenv tqdm scikit-learn sentence-transformers

In [2]:
import sys
import os

# Set HF token from Colab secrets
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(f"HF_TOKEN set: {'yes' if os.environ.get('HF_TOKEN') else 'NO — add it to Colab secrets!'}")

# Add src/ to path if needed
if 'src' not in sys.path:
    sys.path.insert(0, '.')

HF_TOKEN set: yes


In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 1. Load MGSM Dataset

In [4]:
import urllib.request

TARGET_LANGUAGES = ['en', 'zh', 'es', 'bn', 'sw']

# MGSM data from Google Research (simple TSV: question \t numeric_answer)
MGSM_BASE_URL = "https://raw.githubusercontent.com/google-research/url-nlp/main/mgsm/mgsm_{lang}.tsv"

def load_mgsm_from_source(lang):
    """Load MGSM test set directly from Google Research GitHub."""
    url = MGSM_BASE_URL.format(lang=lang)
    response = urllib.request.urlopen(url)
    lines = response.read().decode('utf-8').strip().split('\n')

    examples = []
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        question = parts[0].strip()
        try:
            answer_number = float(parts[1].strip().replace(',', ''))
        except ValueError:
            answer_number = None
        examples.append({
            'question': question,
            'answer_number': answer_number,
        })
    return examples

mgsm_data = {}
for lang in TARGET_LANGUAGES:
    examples = load_mgsm_from_source(lang)
    mgsm_data[lang] = examples
    print(f"{lang}: {len(examples)} examples")
    print(f"  Q: {examples[0]['question'][:80]}...")
    print(f"  A: {examples[0]['answer_number']}")
    print()

en: 250 examples
  Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning an...
  A: 18.0

zh: 250 examples
  Q: 珍妮特的鸭子每天下 16 颗蛋。她每天早上早餐时吃 3 颗，每天用 4 颗为自己的朋友做松饼。剩下的鸭蛋她每天拿去农贸市场卖，每颗新鲜鸭蛋卖 2 美元。她每天在...
  A: 18.0

es: 250 examples
  Q: Los patos de Janet ponen 16 huevos por día. Ella come tres en el desayuno todas ...
  A: 18.0

bn: 250 examples
  Q: জেনেটের হাঁসগুলি প্রতিদিন 16টি করে ডিম পাড়ে। তিনি প্রতিদিন প্রাতরাশে তিনটি করে ড...
  A: 18.0

sw: 250 examples
  Q: Bata wa Janet hutaga mayai 16 kila siku. Huwa anakula matatu wakati wa staftahi ...
  A: 18.0



In [5]:
# Verify the same problem appears in all languages (problem 0)
print("Problem 0 across languages:")
for lang in TARGET_LANGUAGES:
    print(f"\n[{lang}] answer={mgsm_data[lang][0]['answer_number']}")
    print(f"  {mgsm_data[lang][0]['question'][:120]}")

Problem 0 across languages:

[en] answer=18.0
  Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every da

[zh] answer=18.0
  珍妮特的鸭子每天下 16 颗蛋。她每天早上早餐时吃 3 颗，每天用 4 颗为自己的朋友做松饼。剩下的鸭蛋她每天拿去农贸市场卖，每颗新鲜鸭蛋卖 2 美元。她每天在农贸市场赚多少钱？

[es] answer=18.0
  Los patos de Janet ponen 16 huevos por día. Ella come tres en el desayuno todas las mañanas y usa cuatro para hornear ma

[bn] answer=18.0
  জেনেটের হাঁসগুলি প্রতিদিন 16টি করে ডিম পাড়ে। তিনি প্রতিদিন প্রাতরাশে তিনটি করে ডিম খান এবং বন্ধুদের জন্য প্রতিদিন চারটি 

[sw] answer=18.0
  Bata wa Janet hutaga mayai 16 kila siku. Huwa anakula matatu wakati wa staftahi kila asubuhi na huokea marafiki zake maf


## 2. Load Gemma 3 4B IT

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'google/gemma-2-9b-it'
token = os.environ.get('HF_TOKEN')
assert token, "Set HF_TOKEN environment variable first!"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)

print("Loading model (BF16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=token,
)
model.eval()

In [14]:
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Model device: {model.device}")
print(f"Number of layers: {len(model.model.layers)}")
print(f"Hidden size: {model.config.hidden_size}")

Model loaded. Parameters: 9.24B
Model device: cuda:0
Number of layers: 42
Hidden size: 3584


In [15]:
# Quick test: generate on a simple English MGSM problem
test_q = mgsm_data['en'][0]['question']
test_gold = mgsm_data['en'][0]['answer_number']

messages = [
    {'role': 'user', 'content': test_q},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"Prompt ({len(prompt)} chars):")
print(prompt[:300])
print("...")

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    gen_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

output = tokenizer.decode(gen_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\nModel output:\n{output}")
print(f"\nGold answer: {test_gold}")

Prompt (340 chars):
<bos><start_of_turn>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' ma
...

Model output:
Here's how to solve the problem step-by-step:

1. **Calculate the total eggs used:** Janet uses 3 eggs for breakfast + 4 eggs for muffins = 7 eggs per day.

2. **Calculate the number of eggs sold:** She has 16 eggs - 7 eggs used = 9 eggs to sell.

3. **Calculate her earnings:** She sells 9 eggs * $2 per egg = $18.


**Answer:** Janet makes $18 every day at the farmers' market.

Gold answer: 18.0


In [16]:
# Test answer parsing
import re

def parse_answer_number(text):
    """Extract final numeric answer from model output."""
    text = text.strip()
    # Pattern: #### <number>
    match = re.search(r'####\s*([\d,]+(?:\.\d+)?)', text)
    if match:
        return float(match.group(1).replace(',', ''))
    # Pattern: "The answer is <number>"
    for pattern in [
        r'[Tt]he answer is\s*([\d,]+(?:\.\d+)?)',
        r'[Aa]nswer[:\s]*([\d,]+(?:\.\d+)?)',
    ]:
        match = re.search(pattern, text)
        if match:
            return float(match.group(1).replace(',', ''))
    # Fallback: last number
    numbers = re.findall(r'[\d,]+(?:\.\d+)?', text)
    if numbers:
        return float(numbers[-1].replace(',', ''))
    return None

parsed = parse_answer_number(output)
print(f"Parsed: {parsed}, Gold: {test_gold}, Correct: {parsed == test_gold}")

Parsed: 18.0, Gold: 18.0, Correct: True


## 3. Load Gemma Scope 2 SAE

In [17]:
from sae_lens import SAE

# Gemma Scope 2 residual-stream SAEs for Gemma 3 4B IT
# Release name for SAELens (NOT the HuggingFace repo_id)
SAE_RELEASE = 'gemma-scope-9b-it-res'

# sae_id format: layer_{N}_width_{W}_l0_{SIZE}
# Available at subset layers {9, 17, 22, 29}:
#   Widths: 16k, 65k, 262k
#   L0 targets: small, medium, big
# Available at all layers (use release "gemma-scope-2-4b-it-res-all"):
#   Width: 16k

print(f"SAELens release: {SAE_RELEASE}")
print(f"sae_id format: layer_{{N}}_width_{{W}}_l0_{{SIZE}}")
print(f"Example: layer_9_width_16k_l0_medium")

SAELens release: gemma-scope-9b-it-res
sae_id format: layer_{N}_width_{W}_l0_{SIZE}
Example: layer_9_width_16k_l0_medium


In [19]:
# List available SAEs in the release
from huggingface_hub import list_repo_tree

# SAELens release name vs HuggingFace repo_id are different things
SAE_HF_REPO = 'google/gemma-scope-9b-it-res'

try:
    files = list(list_repo_tree(SAE_HF_REPO, repo_type='model', recursive=True))
    # Show directory structure (layer_X/width_Y)
    dirs = set()
    for f in files:
        parts = f.path.split('/')
        if len(parts) >= 2:
            dirs.add('/'.join(parts[:2]))
    print(f"Available SAE directories in {SAE_HF_REPO}:")
    for d in sorted(dirs):
        print(f"  {d}")
except Exception as e:
    print(f"Could not list repo: {e}")
    print(f"Try browsing https://huggingface.co/{SAE_HF_REPO} manually")

Available SAE directories in google/gemma-scope-9b-it-res:
  layer_20/width_131k
  layer_20/width_16k
  layer_31/width_131k
  layer_31/width_16k
  layer_9/width_131k
  layer_9/width_16k


In [20]:
from huggingface_hub import list_repo_tree

files = list(list_repo_tree('google/gemma-scope-9b-it-res', 'layer_9/width_16k', repo_type='model'))
for f in files:
    print(f.path)

layer_9/width_16k/average_l0_14
layer_9/width_16k/average_l0_186
layer_9/width_16k/average_l0_26
layer_9/width_16k/average_l0_47
layer_9/width_16k/average_l0_88


In [21]:
# Load a 16k SAE at layer 9 with medium sparsity
sae_id = 'layer_9_width_16k_l0_medium'
print(f"Loading SAE: release='{SAE_RELEASE}', sae_id='{sae_id}'")

try:
    sae, cfg_dict, sparsity = SAE.from_pretrained(
    release='gemma-scope-9b-it-res-canonical',  # note the -canonical suffix
    sae_id='layer_9/width_16k/canonical',
    )
    print(f"\nSAE loaded successfully!")
    print(f"  d_in (model dim): {sae.cfg.d_in}")
    print(f"  d_sae (features): {sae.cfg.d_sae}")
    print(f"  W_enc shape: {sae.W_enc.shape}")
    print(f"  W_dec shape: {sae.W_dec.shape}")
except Exception as e:
    print(f"Failed to load SAE: {e}")
    print("\nTrying alternative sae_id formats...")
    for alt_id in [
        'layer_9_width_16k_l0_small',
        'layer_9_width_16k_l0_big',
        'layer_9_width_65k_l0_medium',
    ]:
        try:
            sae, cfg_dict, sparsity = SAE.from_pretrained(
                release=SAE_RELEASE,
                sae_id=alt_id,
                device='cuda',
            )
            print(f"  SUCCESS with sae_id='{alt_id}'")
            print(f"  d_sae={sae.cfg.d_sae}, W_dec={sae.W_dec.shape}")
            sae_id = alt_id
            break
        except Exception as e2:
            print(f"  Failed: {alt_id} -> {e2}")

Loading SAE: release='gemma-scope-9b-it-res', sae_id='layer_9_width_16k_l0_medium'


layer_9/width_16k/average_l0_88/params.n(…):   0%|          | 0.00/470M [00:00<?, ?B/s]


SAE loaded successfully!
  d_in (model dim): 3584
  d_sae (features): 16384
  W_enc shape: torch.Size([3584, 16384])
  W_dec shape: torch.Size([16384, 3584])


/tmp/ipykernel_1224/1769046628.py:6: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [25]:
# Test SAE encoding/decoding
if 'sae' in dir():
    # Use the SAE's own declared input dim + dtype — don't hardcode
    d_in = sae.cfg.d_in
    sae_dtype = next(sae.parameters()).dtype
    sae_device = next(sae.parameters()).device

    dummy = torch.randn(1, d_in, device=sae_device, dtype=sae_dtype)

    with torch.no_grad():
        features = sae.encode(dummy)
        reconstruction = sae.decode(features)

    print(f"Input shape: {dummy.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Reconstruction shape: {reconstruction.shape}")
    print(f"Active features (nonzero): {(features > 0).sum().item() / features.shape[0]:.1f}")
    print(f"Reconstruction MSE: {((dummy - reconstruction) ** 2).mean().item():.6f}")
else:
    print("SAE not loaded — fix the loading step above first.")


print(f"Average L0 (active features per token): {(features > 0).float().sum(-1).mean().item():.1f}")

Input shape: torch.Size([1, 3584])
Features shape: torch.Size([1, 16384])
Reconstruction shape: torch.Size([1, 3584])
Active features (nonzero): 126.0
Reconstruction MSE: 1.225367
Average L0 (active features per token): 126.0


## 4. Test nnsight Activation Extraction

In [39]:
from nnsight import NNsight
# Sanity check: verify nnsight activation extraction works across all three SAE layers
nn_model = NNsight(model)

test_input = tokenizer("What is 2 + 3?", return_tensors='pt').to(model.device)

with nn_model.trace(**test_input, scan=False, validate=False):
    resid_9  = nn_model.model.layers[9].output[0].save()
    resid_20 = nn_model.model.layers[20].output[0].save()
    resid_31 = nn_model.model.layers[31].output[0].save()

# Drop .value if you're on nnsight >= 0.3
print(f"Layer 9  residual shape: {resid_9.shape}")   # (1, seq_len, 3584)
print(f"Layer 20 residual shape: {resid_20.shape}")
print(f"Layer 31 residual shape: {resid_31.shape}")

print(f"\nActivation norms:")
print(f"  Layer 9:  {resid_9.float().norm(dim=-1).mean().item():.2f}")
print(f"  Layer 20: {resid_20.float().norm(dim=-1).mean().item():.2f}")
print(f"  Layer 31: {resid_31.float().norm(dim=-1).mean().item():.2f}")

Layer 9  residual shape: torch.Size([9, 3584])
Layer 20 residual shape: torch.Size([9, 3584])
Layer 31 residual shape: torch.Size([9, 3584])

Activation norms:
  Layer 9:  310.02
  Layer 20: 831.75
  Layer 31: 1269.33


In [38]:
long_text = """The quick brown fox jumps over the lazy dog. Scientists have discovered that sparse autoencoders can help interpret the internal representations of large language models by decomposing activations into a dictionary of features. This approach, pioneered by Anthropic and extended by Google DeepMind, has yielded insights into how neural networks represent concepts."""

test_input = tokenizer(long_text, return_tensors='pt').to(model.device)
print(f"Num tokens: {test_input['input_ids'].shape[1]}")

with nn_model.trace(**test_input, scan=False, validate=False):
    resid_9 = nn_model.model.layers[9].output[0].save()

acts = resid_9.reshape(-1, 3584).to(torch.float32).cuda()
print(f"Shape: {acts.shape}")
print(f"Mean token norm: {acts.norm(dim=-1).mean().item():.2f}")

with torch.no_grad():
    features = sae.encode(acts)
    recon = sae.decode(features)

# Exclude BOS token (often weird)
acts_no_bos = acts[1:]
recon_no_bos = recon[1:]
features_no_bos = features[1:]

mse = ((acts_no_bos - recon_no_bos) ** 2).mean().item()
var = acts_no_bos.var().item()
l0 = (features_no_bos > 0).float().sum(-1).mean().item()

print(f"\nExcluding BOS:")
print(f"L0: {l0:.1f}")
print(f"MSE: {mse:.4f} | Var: {var:.4f} | FVE: {1 - mse/var:.3f}")

Num tokens: 63
Shape: torch.Size([63, 3584])
Mean token norm: 167.14

Excluding BOS:
L0: 118.5
MSE: 1.0465 | Var: 5.8064 | FVE: 0.820


In [35]:
print(sae.cfg.metadata)

SAEMetadata({'sae_lens_version': '6.39.0', 'sae_lens_training_version': None, 'model_name': 'gemma-2-9b-it', 'hook_name': 'blocks.9.hook_resid_post', 'hook_head_index': None, 'prepend_bos': True, 'dataset_path': 'monology/pile-uncopyrighted', 'context_size': 1024, 'neuronpedia_id': 'gemma-2-9b-it/9-gemmascope-res-16k'})


In [36]:
import torch

# Move SAE to GPU and keep it float32
sae = sae.to('cuda')

real_acts = resid_9  # (1, seq_len, 3584), bfloat16 on cuda
real_acts_flat = real_acts.reshape(-1, real_acts.shape[-1]).to(dtype=torch.float32, device='cuda')

with torch.no_grad():
    features = sae.encode(real_acts_flat)
    reconstruction = sae.decode(features)

l0 = (features > 0).float().sum(-1).mean().item()
mse = ((real_acts_flat - reconstruction) ** 2).mean().item()
var = real_acts_flat.var().item()
print(f"L0: {l0:.1f}")
print(f"MSE: {mse:.4f} | Var: {var:.4f} | FVE: {1 - mse/var:.3f}")
print(f"Activation norm (mean): {real_acts_flat.norm(dim=-1).mean().item():.2f}")

L0: 790.6
MSE: 825.0143 | Var: 99.3307 | FVE: -7.306
Activation norm (mean): 310.02


In [37]:
with nn_model.trace(**test_input, scan=False, validate=False):
    layer_out = nn_model.model.layers[9].output.save()

print(type(layer_out))
print(len(layer_out) if hasattr(layer_out, '__len__') else 'not a sequence')
print(layer_out[0].shape if hasattr(layer_out, '__getitem__') else 'N/A')

<class 'torch.Tensor'>
1
torch.Size([9, 3584])


In [28]:
# Feed real residual stream activations through the SAE
real_acts = resid_9  # shape (1, seq_len, 3584), from nnsight trace
real_acts_flat = real_acts.reshape(-1, real_acts.shape[-1]).to(sae_dtype).to(sae_device)

with torch.no_grad():
    features = sae.encode(real_acts_flat)
    reconstruction = sae.decode(features)

l0 = (features > 0).float().sum(-1).mean().item()
mse = ((real_acts_flat - reconstruction) ** 2).mean().item()
var = real_acts_flat.float().var().item()
frac_var_explained = 1 - mse / var

print(f"L0 on real activations: {l0:.1f}")
print(f"MSE: {mse:.4f}")
print(f"Fraction of variance explained: {frac_var_explained:.3f}")

L0 on real activations: 790.6
MSE: 825.0143
Fraction of variance explained: -7.306


In [41]:
# Test: encode real activations through SAE
if 'sae' in dir():
    sae_dtype = next(sae.parameters()).dtype
    last_token_act = resid_9[-1:, :].to(dtype=sae_dtype, device='cuda')  # (1, 3584)

    with torch.no_grad():
        features = sae.encode(last_token_act)

    print(f"Feature activations shape: {features.shape}")
    n_active = (features > 0).sum().item()
    print(f"Active features: {n_active} / {features.shape[-1]} ({100*n_active/features.shape[-1]:.1f}%)")

    top_features = torch.argsort(features[0], descending=True)[:10]
    print(f"\nTop 10 active features:")
    for idx in top_features:
        print(f"  Feature {idx.item()}: activation = {features[0, idx].item():.4f}")
else:
    print("SAE not loaded.")

Feature activations shape: torch.Size([1, 16384])
Active features: 95 / 16384 (0.6%)

Top 10 active features:
  Feature 12085: activation = 53.4983
  Feature 5318: activation = 36.4367
  Feature 9081: activation = 25.0582
  Feature 9540: activation = 18.2780
  Feature 12825: activation = 16.4568
  Feature 7780: activation = 13.9739
  Feature 7408: activation = 13.1698
  Feature 13161: activation = 13.1697
  Feature 2601: activation = 12.5282
  Feature 307: activation = 11.1606


In [31]:
print(sae.cfg)
print(sae.cfg.metadata)
# and
print(dir(sae.cfg.metadata))

JumpReLUSAEConfig(d_in=3584, d_sae=16384, dtype='float32', device='cpu', apply_b_dec_to_input=False, normalize_activations='none', reshape_activations='none', metadata=SAEMetadata({'sae_lens_version': '6.39.0', 'sae_lens_training_version': None, 'model_name': 'gemma-2-9b-it', 'hook_name': 'blocks.9.hook_resid_post', 'hook_head_index': None, 'prepend_bos': True, 'dataset_path': 'monology/pile-uncopyrighted', 'context_size': 1024, 'neuronpedia_id': 'gemma-2-9b-it/9-gemmascope-res-16k'}))
SAEMetadata({'sae_lens_version': '6.39.0', 'sae_lens_training_version': None, 'model_name': 'gemma-2-9b-it', 'hook_name': 'blocks.9.hook_resid_post', 'hook_head_index': None, 'prepend_bos': True, 'dataset_path': 'monology/pile-uncopyrighted', 'context_size': 1024, 'neuronpedia_id': 'gemma-2-9b-it/9-gemmascope-res-16k'})
['__class__', '__contains__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getst

## 5. Quick Baseline: Run on 5 problems per language

In [42]:
# Run model on first 5 problems per language to verify pipeline
N_TEST = 5

results = {}
for lang in TARGET_LANGUAGES:
    predictions = []
    gold_answers = []

    for i in range(N_TEST):
        q = mgsm_data[lang][i]['question']
        gold = mgsm_data[lang][i]['answer_number']
        gold_answers.append(gold)

        messages = [{'role': 'user', 'content': q}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

        output = tokenizer.decode(
            gen_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        parsed = parse_answer_number(output)
        predictions.append(parsed)

        correct = parsed is not None and abs(parsed - gold) < 1e-6
        print(f"[{lang}] Q{i}: gold={gold}, parsed={parsed}, correct={correct}")

    correct_count = sum(
        1 for p, g in zip(predictions, gold_answers)
        if p is not None and abs(p - g) < 1e-6
    )
    results[lang] = correct_count / N_TEST

print("\n=== Quick Baseline Results ===")
for lang, acc in results.items():
    print(f"  {lang}: {acc:.0%} ({int(acc * N_TEST)}/{N_TEST})")
print(f"  Average: {sum(results.values()) / len(results):.0%}")

[en] Q0: gold=18.0, parsed=18.0, correct=True
[en] Q1: gold=3.0, parsed=3.0, correct=True
[en] Q2: gold=70000.0, parsed=70000.0, correct=True
[en] Q3: gold=540.0, parsed=540.0, correct=True
[en] Q4: gold=20.0, parsed=20.0, correct=True
[zh] Q0: gold=18.0, parsed=18.0, correct=True
[zh] Q1: gold=3.0, parsed=3.0, correct=True
[zh] Q2: gold=70000.0, parsed=70000.0, correct=True
[zh] Q3: gold=540.0, parsed=540.0, correct=True
[zh] Q4: gold=20.0, parsed=140.0, correct=False
[es] Q0: gold=18.0, parsed=18.0, correct=True
[es] Q1: gold=3.0, parsed=3.0, correct=True


KeyboardInterrupt: 

## 6. Memory Usage Report

In [ ]:
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"GPU Memory Total:     {total / 1e9:.2f} GB")
    print(f"GPU Memory Free:      {(total - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

## Summary

After running this notebook, confirm:
- [ ] MGSM loads for all 5 languages (250 problems each)
- [ ] Same problems appear in same order across languages
- [ ] Gemma 3 4B IT loads in BF16
- [ ] Model generates reasonable math answers
- [ ] Answer parser extracts correct numbers
- [ ] SAE loads from Gemma Scope 2 (note the exact sae_id that works)
- [ ] SAE encoding produces sparse features
- [ ] nnsight extracts residual stream activations
- [ ] Real activations encode through SAE correctly
- [ ] Memory usage is manageable on A100

**Record the exact working `sae_id` format** — we need this for all subsequent experiments.